# Day 1 — The spatial workflow with Visium

**Dataset:** a public 10x Visium *human lymph node* section (spot-based, whole transcriptome).

Today you take a section from raw counts to clusters placed in tissue space, and meet your first spatial statistics. The workflow reuses the single-cell toolkit and adds a spatial layer on top.

**You will learn to**
- read a spatial section and understand the data object;
- compute, read, and act on quality-control metrics;
- normalise, reduce, cluster, and annotate;
- visualise expression in space and run neighbourhood enrichment and spatially variable gene tests.

Worked cells run top to bottom. Exercises are marked and explore on copies, so you can run the whole notebook first, then experiment.

### Setup (run first)

On the course servers this does nothing, since the packages are pre-installed. On Google Colab or a fresh laptop it installs what is missing. Data downloads automatically the first time it is needed, except the Xenium bundle on Day 2 (see the README).

In [ ]:
import importlib, subprocess, sys
_req = [('scanpy','scanpy'), ('squidpy','squidpy'), ('leidenalg','leidenalg'),
        ('igraph','igraph'), ('pyclustree','pyclustree'),
        ('scikit-image','skimage'), ('imageio','imageio')]
_missing = [pip for pip, mod in _req if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=False)
print('environment ready')

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.titlesize": 11,
    "font.weight": "normal",
    "figure.dpi": 110,
})

def _data_dir():
    env = os.environ.get("SPATIAL_COURSE_DATA")
    if env:
        return Path(env)
    shared = Path("/home/shared/spatial_course_data")
    if shared.exists():
        return shared
    local = Path.home() / "spatial_course_data"
    local.mkdir(parents=True, exist_ok=True)
    return local

DATA_DIR = _data_dir()
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

def load_visium():
    p = DATA_DIR / "visium_lymph_node.h5ad"
    if p.exists():
        adata = sc.read_h5ad(p)
    else:
        adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
        try:
            adata.write(p)
        except Exception:
            pass
    adata.var_names_make_unique()
    return adata

def load_seqfish():
    p = DATA_DIR / "seqfish_embryo.h5ad"
    if p.exists():
        return sc.read_h5ad(p)
    adata = sq.datasets.seqfish()
    try:
        adata.write(p)
    except Exception:
        pass
    return adata

def xenium_dir():
    d = DATA_DIR / "xenium_kidney"
    if (d / "cell_feature_matrix.h5").exists():
        return d
    raise FileNotFoundError(
        f"Xenium bundle not found in {d}. See the README (Data): download the 10x "
        "Xenium Output Bundle once and unzip it there, or run scripts/prefetch_data.py."
    )

print("data dir:", DATA_DIR)

## 1. The data object

> **Method note — AnnData vs SpatialData.** A single `AnnData` holds the count matrix (`X`), per-spot and per-gene tables (`obs`, `var`), coordinates (`obsm['spatial']`), and the tissue image (`uns['spatial']`). For multi-sample projects or when you need images and shapes together, the `spatialdata` framework wraps several AnnData tables with aligned images; we stay with plain AnnData here because it is lighter and enough for one section.

We load a prepared `.h5ad` from the shared folder. To read a raw Space Ranger output folder instead you would call `sq.read.visium(path)`.

In [ ]:
adata = load_visium()
adata

## 2. Quality control

> **Method note — what to measure.** Total counts and genes per spot show tissue coverage and flag spots that are mostly background. Mitochondrial fraction marks stressed regions. Always view each metric *in space*: a bad area is usually a contiguous patch (an edge, a fold), not scattered spots.

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3))
axs[0].hist(adata.obs['total_counts'], bins=60); axs[0].set_title('total_counts')
axs[1].hist(adata.obs['n_genes_by_counts'], bins=60); axs[1].set_title('n_genes_by_counts')
axs[2].hist(adata.obs['pct_counts_mt'], bins=60); axs[2].set_title('pct_counts_mt')
plt.tight_layout(); plt.show()

In [ ]:
sc.pl.spatial(adata, color=['total_counts', 'n_genes_by_counts'], size=1.4)

> **Exercise 1.** Pick a minimum-counts threshold from the plots above. Work on a copy called `adata_q`, filter it, and report how many spots remain and how many were dropped. There is no single right answer; be ready to justify yours.

In [ ]:
# your code here


## 3. Filter (worked spine)

We apply a moderate filter to the working object. Adjust to match what you saw.

In [ ]:
sc.pp.filter_cells(adata, min_counts=500)
sc.pp.filter_genes(adata, min_cells=10)
adata

## 4. Normalisation

> **Method note — what normalisation does, and how to choose.**
> Total-count normalisation then `log1p` rescales every spot to a common depth and compresses the dynamic range,
> $$\tilde{x}_{ig} = \log\left(1 + \frac{x_{ig}}{\sum_{g'} x_{ig'}}\,s\right),$$
> with target sum $s$ (median depth or $10^4$). Fast and fine as a first pass, but the variance of $\tilde{x}$ still grows with the mean, so a few highly expressed genes dominate PCA.
>
> **Analytic Pearson residuals** model counts as negative binomial with expected value $\mu_{ig} = \frac{(\sum_{g'} x_{ig'})(\sum_{i'} x_{i'g})}{\sum_{i'g'} x_{i'g'}}$ and a shared overdispersion $\theta$, and return the variance-stabilised residual
> $$r_{ig} = \frac{x_{ig}-\mu_{ig}}{\sqrt{\mu_{ig} + \mu_{ig}^2/\theta}},\qquad \text{clipped to } \pm\sqrt{n}.$$
> This breaks the mean-variance coupling and can sharpen rare populations; it assumes the NB model with one $\theta$ holds, and can over-weight rare genes when it does not. SCTransform is a related regression-based variant.
>
> For spots, depth partly reflects how many cells were captured, so normalisation also removes some cell-density signal. We keep raw counts in a layer so we can switch.

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

> **Exercise 2.** On a copy of the raw counts, try analytic Pearson residuals and compute highly variable genes from them. How much does the top-2000 HVG set overlap with the log-normalised one?

In [ ]:
# your code here


## 5. Features, embedding, clustering

> **Method note — each choice and what it assumes.**
> - **`n_top_genes` (highly variable genes):** keep the most variable genes, assuming signal lives in variable genes and the rest is noise. Too few drops subtypes; too many re-adds noise. 2000 is a common default.
> - **`n_comps` (PCA):** PCA assumes the leading axes of variance are biological. Keep enough components to capture structure (an elbow in the explained variance) but not so many that noise dominates; 30–50 is typical.
> - **`n_neighbors`:** the local neighbourhood size for the kNN graph. Small values stress fine local structure, large values smooth toward global structure, and this propagates into both UMAP and clustering.
> - **`resolution` (Leiden):** Leiden maximises a modularity-type objective
> $$Q = \frac{1}{2m}\sum_{ij}\left(A_{ij} - \gamma\,\frac{k_i k_j}{2m}\right)\delta(c_i,c_j),$$
> with adjacency $A$, degrees $k_i$, edge count $m$, and resolution $\gamma$. Higher $\gamma$ gives more, smaller clusters. There is no ground-truth resolution, so treat clustering as exploratory: scan a few values and pick the grain that matches the biology. The `igraph` flavour is the fast, recommended backend.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_neighbors=15)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1.0, flavor='igraph', n_iterations=2, directed=False)

In [ ]:
sc.pl.umap(adata, color=['leiden', 'total_counts'])
sc.pl.spatial(adata, color='leiden', size=1.4)

## 6. Choosing the clustering resolution

> **Method note — make the resolution choice visible.** There is no true resolution. A practical way to choose is a *clustering tree* (clustree): cluster at a range of resolutions and draw how cells flow from coarse to fine clusters. Branches that persist across resolutions are robust populations; a node with many incoming edges, or clusters that keep shattering and reshuffling, signals over-clustering. Read the tree together with the number of clusters per resolution and pick a value just before the count explodes or the tree turns messy. We use `pyclustree`, the Python port of the R `clustree`.

In [ ]:
resolutions = [round(r, 1) for r in np.arange(0.2, 1.5, 0.2)]
for r in resolutions:
    key = f"leiden_{str(r).replace('.', '_')}"
    sc.tl.leiden(adata, resolution=r, flavor='igraph', n_iterations=2, directed=False, key_added=key)
counts = [adata.obs[f"leiden_{str(r).replace('.', '_')}"].nunique() for r in resolutions]
pd.DataFrame({'resolution': resolutions, 'n_clusters': counts})

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(resolutions, counts, marker='o')
plt.xlabel('resolution'); plt.ylabel('number of clusters'); plt.title('clusters vs resolution')
plt.tight_layout(); plt.show()

In [ ]:
keys = [f"leiden_{str(r).replace('.', '_')}" for r in resolutions]
fig = None
try:
    from pyclustree import clustree
    fig = clustree(adata, keys, show_fraction=True)
    fig.set_size_inches(9, 11)
except Exception as e:
    print('clustree unavailable:', e, '- install with: pip install pyclustree')
fig

> **Exercise (resolution).** From the tree and the count curve, choose a resolution. Set `adata.obs['leiden']` to that clustering and note how many clusters it has. Everything downstream uses `adata.obs['leiden']`.

In [ ]:
# your code here


## 7. Marker genes and annotation

> **Method note — naming clusters.** Marker-based naming (rank genes, match to known markers) is transparent and works without a reference. When a matched scRNA-seq reference exists, label transfer (`sc.tl.ingest`, celltypist, scANVI) scales better. We use markers here.

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=8, sharey=False)

In [ ]:
markers = [g for g in ['MS4A1', 'CD3D', 'CD8A', 'LYZ', 'CR2'] if g in adata.var_names]
sc.pl.spatial(adata, color=markers, size=1.4)

> **Exercise 3.** Choose one cluster, propose a label from its markers, and decide from the spatial plot whether it forms a coherent region or is dispersed. Highlight that cluster in space to support your call.

In [ ]:
# your code here


## 8. Spatial visualization

> **Method note — read the right map.** Space is the point of these data, so most questions are answered by plotting a value back onto the tissue. A few habits help: choose a **continuous** overlay (a gene, a QC metric) to see gradients and intensity, and a **categorical** overlay (clusters, types) to see compartments and boundaries; tune spot `size` and `alpha` until structure is legible without spots smearing into one another; **crop** to a region to inspect fine layout; and use `groups=` to spotlight one or two categories against the rest. What you are looking for: contiguous *compartments* (a connected patch), smooth *gradients* (a continuous ramp), or *scattered* signal (a state or a technical artefact, not a place).

In [ ]:
viz_genes = [g for g in ['MS4A1', 'CD3D', 'CD8A', 'LYZ', 'CR2'] if g in adata.var_names]
sc.pl.spatial(adata, color=viz_genes[0], size=1.6)

In [ ]:
sc.pl.spatial(adata, color=viz_genes[:4], size=1.4)

In [ ]:
sc.pl.spatial(adata, color=['leiden', 'total_counts'], size=1.4)

In [ ]:
sc.pl.spatial(adata, color='leiden', size=0.8, alpha=0.5, legend_loc=None)
sc.pl.spatial(adata, color='leiden', size=2.2, alpha=1.0, legend_loc=None)

In [ ]:
xy = adata.obsm['spatial']
x0, y0 = xy.min(0)
x1, y1 = xy.max(0)
crop = (x0 + 0.35 * (x1 - x0), x0 + 0.7 * (x1 - x0), y0 + 0.35 * (y1 - y0), y0 + 0.7 * (y1 - y0))
sc.pl.spatial(adata, color='leiden', crop_coord=crop, size=2.6)

In [ ]:
top_clusters = adata.obs['leiden'].value_counts().index[:2].tolist()
sc.pl.spatial(adata, color='leiden', groups=top_clusters, size=1.4)

> **Exercise (visualization).** Choose four genes and plot them as a grid; crop to a region you find interesting and describe it; pick a `size`/`alpha` that makes the structure legible; and overlay a single cluster on the tissue.

In [ ]:
# your code here


## 9. First spatial statistics

> **Method note — graph, enrichment, autocorrelation.**
> The spatial graph with weights $w_{ij}$ defines neighbours; for Visium the hexagonal grid (6 neighbours) is natural, and every statistic below is relative to it.
>
> **Neighbourhood enrichment** counts edges between each pair of cluster types, permutes the labels to build a null, and reports $z = (O - \mu_\pi)/\sigma_\pi$, where $O$ is the observed count and $\mu_\pi,\sigma_\pi$ come from permutations. Permuting labels keeps composition but breaks spatial structure, so $z$ measures co-location, not abundance.
>
> **Spatial autocorrelation** ranks genes that vary smoothly across the tissue. Moran's I,
> $$I = \frac{N}{W}\,\frac{\sum_{ij} w_{ij}(x_i-\bar{x})(x_j-\bar{x})}{\sum_i (x_i-\bar{x})^2},\qquad W=\sum_{ij} w_{ij},$$
> lies in roughly $[-1,1]$ with null expectation $-1/(N-1)$; $I>0$ marks smooth gradients, $I<0$ a checkerboard. Geary's C,
> $$C = \frac{N-1}{2W}\,\frac{\sum_{ij} w_{ij}(x_i-x_j)^2}{\sum_i (x_i-\bar{x})^2},$$
> lies in roughly $[0,2]$ with $C<1$ for positive autocorrelation, and reacts more to local differences. The two can disagree, which is itself informative.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type='grid', n_neighs=6)
sq.gr.nhood_enrichment(adata, cluster_key='leiden')
sq.pl.nhood_enrichment(adata, cluster_key='leiden', figsize=(5, 5))

> **Reading the enrichment map.** The heatmap is cluster-by-cluster, coloured by the z-score. The diagonal shows self-aggregation: a strongly positive diagonal means a cluster forms compact patches rather than scattering. Off the diagonal, a positive (warm) z means two clusters are neighbours more than chance (co-location, e.g. a layered boundary or an interface), and a negative (cool) z means they avoid each other (segregated compartments). The magnitude is a signal-to-noise measure from the permutations, not a probability, so read the overall pattern and then confirm a striking pair by plotting both clusters in space. Because the test uses this graph and this section's composition, it describes the sample, not a universal rule.

In [ ]:
hvg = adata.var_names[adata.var['highly_variable']].tolist()
sq.gr.spatial_autocorr(adata, mode='moran', genes=hvg)
adata.uns['moranI'].head(10)

> **Reading Moran's I.** The table is sorted by `I`. Genes near the top (towards 1) have smooth spatial structure: high-expressing spots group into regions or gradients. Genes near the null expectation $-1/(N-1)$ (about 0) have no spatial pattern beyond chance. squidpy also returns p-values; use the FDR-corrected column (`pval_norm_fdr_bh`) to call significance rather than `I` alone, since on a large section a modest `I` can still be highly significant. Treat a high-`I` gene as a hypothesis: plot it in space, ask what structure it traces (a compartment, a gradient, a vessel), and cross-check against your clusters and domains.

In [ ]:
top = adata.uns['moranI'].head(4).index.tolist()
sc.pl.spatial(adata, color=top, size=1.4)

> **Exercise 4.** Compute Geary's C on the same gene set and compare its top genes with Moran's. Do the two rankings agree, and where do they differ?

In [ ]:
# your code here


## 10. Spatial domains

> **Method note — domains vs clusters.** The Leiden clusters above group spots by *expression alone*; they ignore where a spot sits. A **spatial domain** is a spatially coherent region — a tissue compartment such as a follicle or a T-cell zone — so two spots in the same domain should be both transcriptionally and physically close. A transparent way to get domains (the *nichePCA* / BANKSY idea) is to give each spot a little of its neighbourhood before clustering. Build the spatial graph, row-normalise its adjacency to weights $w_{ij}$ with $\sum_j w_{ij}=1$, average expression over neighbours, and stack it onto the spot's own expression as an augmented feature
> $$h_i = \big[\,x_i,\ \lambda\,\bar{x}_{\mathcal N(i)}\,\big],\qquad \bar{x}_{\mathcal N(i)} = \sum_j w_{ij}\,x_j,$$
> with a mixing weight $\lambda\ge 0$. PCA on $h$, then neighbours and Leiden as usual, yields `domain` labels. $\lambda=0$ recovers the expression clustering; larger $\lambda$ smooths labels into contiguous regions at the cost of fine within-region detail.
>
> Dedicated tools formalise this: BayesSpace (Zhao et al., *Nat Biotechnol* 2021) and SpaGCN (Hu et al., *Nat Methods* 2021) add a spatial prior or a graph-convolutional model; STAGATE (Dong & Zhang, *Nat Commun* 2022) and GraphST (Long et al., *Nat Commun* 2023) use graph autoencoders; BANKSY (Singhal et al., *Nat Genet* 2024) augments features with neighbour means and azimuthal gradients much like here. Transparent feature-stacking is easy to reason about; the learned methods can capture subtler domains but add assumptions and compute.

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
def spatial_domains(adata, lam, n_comps=30, resolution=1.0, seed=0):
    sq.gr.spatial_neighbors(adata, coord_type='grid', n_neighs=6)
    W = normalize(adata.obsp['spatial_connectivities'], norm='l1', axis=1)
    hvg = adata.var_names[adata.var['highly_variable']]
    Xh = adata[:, hvg].X
    Xh = Xh.toarray() if hasattr(Xh, 'toarray') else np.asarray(Xh)
    H = np.concatenate([Xh, lam * (W @ Xh)], axis=1)
    Hp = PCA(n_components=n_comps, random_state=seed).fit_transform(H)
    tmp = sc.AnnData(Hp.astype('float32'))
    sc.pp.neighbors(tmp, n_neighbors=15)
    sc.tl.leiden(tmp, resolution=resolution, flavor='igraph', n_iterations=2, directed=False, random_state=seed)
    return tmp.obs['leiden'].to_numpy()

In [ ]:
adata.obs['domain'] = pd.Categorical(spatial_domains(adata, lam=0.8))
print(adata.obs['domain'].nunique(), 'domains')
sc.pl.spatial(adata, color=['leiden', 'domain'], size=1.4)

In [ ]:
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(adata.obs['leiden'], adata.obs['domain'])
print('adjusted Rand index (leiden vs domain):', round(ari, 3))

> **Reading the comparison.** Side by side, the `domain` map should look smoother than `leiden`: salt-and-pepper spots get absorbed into the region around them. The adjusted Rand index measures how much the two partitions agree (1 = identical, 0 = chance); a moderate value means smoothing has genuinely regrouped spots, not just relabelled them. That is the trade-off — spatial coherence for fine detail.

> **Exercise (domains).** Recompute domains for a few values of the neighbour weight $\lambda$ (e.g. 0, 1, 2) and plot them in space. How does the map change as $\lambda$ grows, and which value gives compartments you would defend?

In [ ]:
# your code here


## 11. Cell-type deconvolution

> **Method note — a spot is a mixture.** Each Visium spot (55 µm) covers several cells, so its profile is a blend of cell types. *Deconvolution* estimates the per-spot proportions. Given a signature matrix $S$ (genes $\times$ types, the reference mean profile of each type) and a spot $y$ (expression over the same genes), the simplest estimator solves a non-negative least squares problem
> $$\hat{x} = \arg\min_{x\ge 0}\lVert S x - y\rVert_2^2,$$
> then normalises $\hat{x}$ to sum to one. Non-negativity is the only biology imposed; there is no count model and no platform correction, which is exactly why it is a baseline rather than a production method.
>
> Production methods model counts and platform effects: RCTD (Cable et al., *Nat Biotechnol* 2022), cell2location (Kleshchevnikov et al., *Nat Biotechnol* 2022), Tangram (Biancalani et al., *Nat Methods* 2021), stereoscope (Andersson et al., *Commun Biol* 2020), SPOTlight (Elosua-Bayes et al., *NAR* 2021), and DestVI (Lopez et al., *Nat Biotechnol* 2022). For an independent comparison see the benchmark of Li et al., *Nat Commun* 2023 ("Benchmarking spatial and single-cell transcriptomics integration methods").
>
> **Reference, honestly.** Good deconvolution needs a *matched external* scRNA-seq reference. To keep this notebook self-contained and offline we build a **teaching stand-in**: per-type signatures averaged from marker-positive spots *in this same section*. That is partly circular (the signatures come from the data we deconvolve), so read the proportions as a mechanics demo, not a result. Swapping in a real reference's per-type means for `S` is the only change needed.

In [ ]:
from scipy.optimize import nnls
lymph_markers = {
    'B': ['MS4A1', 'CD79A', 'CD79B', 'CR2'],
    'T': ['CD3D', 'CD3E', 'CD2', 'IL7R'],
    'CD8_T': ['CD8A', 'CD8B', 'GZMK'],
    'Myeloid': ['LYZ', 'CD68', 'ITGAX'],
    'Plasma': ['MZB1', 'IGHG1', 'XBP1'],
    'Endothelial': ['PECAM1', 'VWF', 'CLDN5'],
}
lymph_markers = {k: [g for g in v if g in adata.var_names] for k, v in lymph_markers.items()}
lymph_markers = {k: v for k, v in lymph_markers.items() if v}
types = list(lymph_markers)
for ct, genes in lymph_markers.items():
    sc.tl.score_genes(adata, genes, score_name=f'sig_{ct}')
types

In [ ]:
genes_use = adata.var_names[adata.var['highly_variable']].tolist()
Xg = adata[:, genes_use].X
Xg = np.asarray(Xg.toarray() if hasattr(Xg, 'toarray') else Xg, dtype='float64')
S = np.vstack([Xg[(adata.obs[f'sig_{ct}'] >= adata.obs[f'sig_{ct}'].quantile(0.9)).to_numpy()].mean(0)
               for ct in types]).T
print('signature matrix S:', S.shape, '(genes x types)')

In [ ]:
props = np.zeros((adata.n_obs, len(types)))
for i in range(adata.n_obs):
    x, _ = nnls(S, Xg[i])
    total = x.sum()
    props[i] = x / total if total > 0 else x
adata.obsm['deconv'] = props
for j, ct in enumerate(types):
    adata.obs[f'prop_{ct}'] = props[:, j]
print('proportions stored in adata.obsm["deconv"]')

In [ ]:
sc.pl.spatial(adata, color=[f'prop_{ct}' for ct in types[:2]], size=1.4)

In [ ]:
adata.obs.groupby('domain', observed=True)[[f'prop_{ct}' for ct in types]].mean().round(2)

> **Reading the proportions.** Each map shows where a type concentrates; the per-domain table summarises composition by region (from §10). In a lymph node you would expect B-cell-rich follicles and T-cell-rich interfollicular zones to fall into different domains. Because the signatures came from this section, treat the split as illustrative — a real reference is what makes proportions trustworthy.

> **Exercise (deconvolution).** Compare mean proportions across the spatial domains and say which type dominates each. Then name two reasons NNLS can be unreliable here.

In [ ]:
# your code here


## 12. Image annotation transfer (napari)

> **Method note — bring the histology onto the spots.** Often the most reliable region labels come from the *image*, drawn by eye on the H&E: follicle, paracortex, medulla, capsule. The plan is: export the tissue image, paint regions **outside** the notebook, then transfer the painted labels back onto the spots by pixel lookup.
>
> **napari is a GUI** and does not run in a headless notebook — the painting step happens in a desktop session (napari, or QuPath / Fiji / labelme). The notebook does the two parts that *are* programmatic: the export and the transfer. (napari can also be scripted via `napari.Viewer()` when a display is available, and `napari-spatialdata` gives an integrated route on a `SpatialData` object locally.)
>
> Coordinates: `adata.obsm['spatial']` is in full-resolution image pixels, and the hires image is scaled by `tissue_hires_scalef`, so a spot's hires pixel is `obsm['spatial'] * scalef` (column = x, row = y).

In [ ]:
try:
    import imageio.v3 as iio
    lib = list(adata.uns['spatial'])[0]
    img = adata.uns['spatial'][lib]['images']['hires']
    scalef = adata.uns['spatial'][lib]['scalefactors']['tissue_hires_scalef']
    rgb = img[..., :3]
    rgb8 = (255 * np.clip(rgb, 0, 1)).astype('uint8') if rgb.max() <= 1.0 else rgb.astype('uint8')
    iio.imwrite(RESULTS / 'visium_hires.png', rgb8)
    print('hires image', img.shape, '| tissue_hires_scalef', round(float(scalef), 4))
    print('wrote', RESULTS / 'visium_hires.png')
except Exception as e:
    print('image export skipped:', e)

**Paint, then return.** Open `results/visium_hires.png` in napari (or QuPath / Fiji / labelme), paint each region with a distinct integer label, and export a label-mask PNG of the **same height and width** to `results/visium_regions_mask.png`. The next cell loads that mask if it exists; otherwise it synthesises a stand-in from the image (Otsu tissue detection plus a left/right split) so the workflow runs end to end without a real annotation.

In [ ]:
try:
    import imageio.v3 as iio
    from skimage.color import rgb2gray
    from skimage.filters import threshold_otsu
    lib = list(adata.uns['spatial'])[0]
    img = adata.uns['spatial'][lib]['images']['hires']
    scalef = adata.uns['spatial'][lib]['scalefactors']['tissue_hires_scalef']
    mask_path = RESULTS / 'visium_regions_mask.png'
    if mask_path.exists():
        mask = iio.imread(mask_path)
        mask = mask[..., 0] if mask.ndim == 3 else mask
        source = 'painted mask'
    else:
        gray = rgb2gray(img[..., :3])
        tissue = gray < threshold_otsu(gray)
        cols = np.arange(gray.shape[1])[None, :]
        mask = np.zeros(gray.shape, dtype='uint8')
        mask[tissue & (cols < gray.shape[1] // 2)] = 1
        mask[tissue & (cols >= gray.shape[1] // 2)] = 2
        iio.imwrite(mask_path, mask)
        source = 'Otsu stand-in'
    coords = adata.obsm['spatial'] * scalef
    cc = np.clip(coords[:, 0].astype(int), 0, mask.shape[1] - 1)
    rr = np.clip(coords[:, 1].astype(int), 0, mask.shape[0] - 1)
    adata.obs['region'] = pd.Categorical(mask[rr, cc].astype(str))
    print('region source:', source)
    print(adata.obs['region'].value_counts())
    sc.pl.spatial(adata, color='region', size=1.4)
except Exception as e:
    print('annotation transfer skipped:', e)

> **Exercise (annotation transfer).** Cross-tabulate the transferred `region` against the spatial domains from §10. Do image regions and molecular domains line up? If you painted a real mask, drop it at `results/visium_regions_mask.png` and re-run the transfer. What would a real histology annotation add over the stand-in?

In [ ]:
# your code here


## 13. Save

In [ ]:
adata.write(RESULTS / 'visium_lymph_node_processed.h5ad')

### Recap

Raw counts to clusters in space, plus neighbourhood enrichment and spatial autocorrelation. Tomorrow: single-cell resolution with Xenium, where segmentation and control metrics need extra care.